# SRΨ-v1.0 单文件版本（解决缓存问题）

**使用方法**：
1. Runtime → Restart runtime
2. 从上到下运行所有 cells
3. 不要跳过任何 cell

In [ ]:
# Step 1: 挂载 Drive 并安装依赖
from google.colab import drive
drive.mount('/content/drive')

!pip install torch torchvision numpy pyyaml tqdm tensorboard -q

import torch
import numpy as np
import os
import sys

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

In [ ]:
# Step 2: 创建所有文件（一次性）
import shutil

# 完全清理
if os.path.exists('/content/srpsi_v1_baseline'):
    shutil.rmtree('/content/srpsi_v1_baseline')

# 创建目录
os.makedirs('/content/srpsi_v1_baseline/src/models', exist_ok=True)
os.makedirs('/content/srpsi_v1_baseline/config', exist_ok=True)
os.chdir('/content/srpsi_v1_baseline')
sys.path.insert(0, '/content/srpsi_v1_baseline')

# ===== 创建数据生成模块 =====
with open('src/data_gen.py', 'w') as f:
    f.write('''import numpy as np

def generate_burgers_1d(num_samples=1000, total_steps=100, nx=128, 
                        dt=0.01, dx=0.01, nu=0.01, seed=42):
    np.random.seed(seed)
    x = np.linspace(0, 2*np.pi, nx, endpoint=False)
    
    data = []
    for _ in range(num_samples):
        A = np.random.uniform(0.5, 1.5)
        k = np.random.uniform(1, 4)
        u = A * np.sin(k * x) + 0.1 * np.random.randn(nx)
        
        trajectory = [u.copy()]
        for _ in range(total_steps - 1):
            u_xx = np.roll(u, -1) - 2*u + np.roll(u, 1)
            u = u - dt * u * np.roll(u, -1) - np.roll(u, 1) / (2*dx) + nu * u_xx / (dx**2)
            trajectory.append(u.copy())
        
        data.append(np.array(trajectory))
    
    return np.array(data)
''')

# ===== 创建数据加载模块 =====
with open('src/datasets.py', 'w') as f:
    f.write('''import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class FieldRolloutDataset(Dataset):
    def __init__(self, array, tin, tout):
        self.array = array
        self.tin = tin
        self.tout = tout
    
    def __len__(self):
        return len(self.array)
    
    def __getitem__(self, idx):
        seq = self.array[idx]
        return {
            'x': torch.tensor(seq[:self.tin], dtype=torch.float32),
            'y': torch.tensor(seq[self.tin:self.tin + self.tout], dtype=torch.float32)
        }

def create_dataloaders(data_path, tin, tout, batch_size, 
                        num_train, num_val, num_test, num_workers=0, seed=42):
    data = np.load(data_path)
    
    train_data = data[:num_train]
    val_data = data[num_train:num_train+num_val]
    test_data = data[num_train+num_val:num_train+num_val+num_test]
    
    train_loader = DataLoader(FieldRolloutDataset(train_data, tin, tout), 
                              batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(FieldRolloutDataset(val_data, tin, tout),
                            batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(FieldRolloutDataset(test_data, tin, tout),
                             batch_size=batch_size, shuffle=False, num_workers=num_workers)
    
    return train_loader, val_loader, test_loader
''')

# ===== 创建模型文件（最新修复版本）=====
with open('src/models/srpsi_engine_tiny.py', 'w') as f:
    f.write('''import torch
import torch.nn as nn

def _init_weights(module):
    if isinstance(module, (nn.Linear, nn.Conv1d)):
        nn.init.xavier_uniform_(module.weight, gain=0.01)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.0)

class InputEncoder(nn.Module):
    def __init__(self, tin, nx, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(tin, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim * 2),
        )
        self.apply(_init_weights)
    
    def forward(self, x):
        x_mean = x.mean(dim=1, keepdim=True)
        x_std = x.std(dim=1, keepdim=True) + 1e-6
        x = (x - x_mean) / x_std
        x = x.transpose(1, 2)
        z = torch.clamp(self.net(x), -10, 10)
        real, imag = z.chunk(2, dim=-1)
        return torch.stack([real, imag], dim=-1)

class SRPsiBlock(nn.Module):
    def __init__(self, hidden_dim, kernel_size=5):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv1d(hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.phase = nn.Parameter(torch.zeros(hidden_dim))
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.Tanh()
        )
        self.apply(_init_weights)
    
    def forward(self, psi):
        real = psi[..., 0].transpose(1, 2)
        imag = psi[..., 1].transpose(1, 2)
        dpsi_real = self.conv(real)
        dpsi_imag = self.conv(imag)
        dpsi = torch.stack([dpsi_real, dpsi_imag], dim=-1).transpose(1, 2)
        
        cos_p = torch.cos(self.phase).unsqueeze(0).unsqueeze(0)
        sin_p = torch.sin(self.phase).unsqueeze(0).unsqueeze(0)
        psi_rot = psi.clone()
        psi_rot[..., 0] = psi[..., 0] * cos_p - psi[..., 1] * sin_p
        psi_rot[..., 1] = psi[..., 0] * sin_p + psi[..., 1] * cos_p
        
        psi_flat = torch.cat([psi_rot[..., 0], psi_rot[..., 1]], dim=-1)
        psi_out = psi + self.proj(psi_flat).view_as(psi)
        
        return psi_out

class OutputDecoder(nn.Module):
    def __init__(self, hidden_dim, nx, tout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.final = nn.Linear(hidden_dim, tout)
        self.tout = tout
        self.nx = nx
        self.apply(_init_weights)
    
    def forward(self, psi):
        B, X, D, _ = psi.shape
        psi_flat = psi.view(B, X, -1)
        h = self.net(psi_flat)
        out = self.final(h)
        return out.transpose(1, 2)  # 关键：使用 transpose

class SRPsiEngineTiny(nn.Module):
    def __init__(self, tin, nx, hidden_dim=64, depth=3, 
                 kernel_size=5, dt=0.01, tout=32):
        super().__init__()
        self.encoder = InputEncoder(tin, nx, hidden_dim)
        self.blocks = nn.ModuleList([SRPsiBlock(hidden_dim, kernel_size) for _ in range(depth)])
        self.decoder = OutputDecoder(hidden_dim, nx, tout)
    
    def forward(self, x):
        psi = self.encoder(x)
        for block in self.blocks:
            psi = block(psi)
        return self.decoder(psi)
''')

with open('src/models/__init__.py', 'w') as f:
    f.write('''from .srpsi_engine_tiny import SRPsiEngineTiny
__all__ = ['SRPsiEngineTiny']
''')

# ===== 创建配置文件 =====
with open('config/burgers.yaml', 'w') as f:
    f.write('''seed: 42
device: cuda

task:
  name: burgers_1d
  nx: 128
  tin: 16
  tout: 32
  dt: 0.01
  dx: 0.01
  samples_train: 4000
  samples_val: 400
  samples_test: 400
  noise_std: 0.0
  nu: 0.01
  domain_size: 6.283185307179586

training:
  batch_size: 32
  epochs: 80
  lr: 0.0001
  weight_decay: 0.00001
  grad_clip: 0.5

loss:
  lambda_cons: 0.1
  lambda_phase: 0.0
  lambda_smooth: 0.02

model:
  hidden_dim: 64
  depth: 3
  kernel_size: 5
  dropout: 0.0

output:
  log_interval: 10
  save_interval: 20
''')

print("✅ 所有文件已创建")

In [ ]:
# Step 3: 验证模型文件
with open('src/models/srpsi_engine_tiny.py', 'r') as f:
    content = f.read()
    
if 'return out.transpose(1, 2)' in content:
    print("✅ 模型文件正确：使用 transpose")
else:
    print("❌ 模型文件错误")
    
if 'dpsi_real = self.conv(real)' in content:
    print("✅ 复数卷积正确：分别处理实部虚部")
else:
    print("❌ 复数卷积错误")

In [ ]:
# Step 4: 生成或加载数据
from src.data_gen import generate_burgers_1d
from datetime import datetime

DATA_DIR = '/content/drive/MyDrive/srpsi-colab-baseline/data'
os.makedirs(DATA_DIR, exist_ok=True)

DATA_PATH = os.path.join(DATA_DIR, 'burgers_1d_v1.npy')

if not os.path.exists(DATA_PATH):
    print("生成数据...")
    data = generate_burgers_1d(
        num_samples=4800,
        total_steps=48,
        nx=128,
        dt=0.01,
        dx=0.01,
        nu=0.01,
        seed=42
    )
    np.save(DATA_PATH, data)
    print(f"✅ 数据已保存: {data.shape}")
else:
    print(f"✅ 数据已存在")
    data = np.load(DATA_PATH)
    print(f"   Shape: {data.shape}")

In [ ]:
# Step 5: 创建并运行训练脚本
train_script = '''import sys
import os
sys.path.insert(0, os.getcwd())

import argparse
import yaml
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from tqdm import tqdm

from src.data_gen import generate_burgers_1d
from src.datasets import create_dataloaders
from src.models.srpsi_engine_tiny import SRPsiEngineTiny

def total_loss(model, x, pred, y, cfg, epoch=0):
    loss_pred = torch.mean((pred - y) ** 2)
    energy_pred = 0.5 * torch.sum(pred ** 2, dim=[1, 2])
    energy_target = 0.5 * torch.sum(y ** 2, dim=[1, 2])
    loss_cons = torch.mean(torch.abs(energy_pred - energy_target))
    lambda_cons = cfg['loss'].get('lambda_cons', 0.1)
    loss_total = loss_pred + lambda_cons * loss_cons
    return loss_total, {'loss_total': loss_total.item()}

def create_model(cfg, device):
    model = SRPsiEngineTiny(
        tin=cfg['task']['tin'],
        nx=cfg['task']['nx'],
        hidden_dim=cfg['model']['hidden_dim'],
        depth=cfg['model']['depth'],
        kernel_size=cfg['model']['kernel_size'],
        dt=cfg['task']['dt'],
        tout=cfg['task']['tout'],
    )
    return model.to(device)

def main():
    import yaml
    cfg = yaml.safe_load(open('config/burgers.yaml'))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    from src.datasets import create_dataloaders
    train_loader, val_loader, test_loader = create_dataloaders(
        data_path='''" + "'" + DATA_PATH + "'" + '''',
        tin=cfg['task']['tin'],
        tout=cfg['task']['tout'],
        batch_size=cfg['training']['batch_size'],
        num_train=cfg['task']['samples_train'],
        num_val=cfg['task']['samples_val'],
        num_test=cfg['task']['samples_test'],
        num_workers=0
    )
    
    model = create_model(cfg, device)
    optimizer = optim.Adam(model.parameters(), lr=cfg['training']['lr'])
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Starting training...")
    
    for epoch in range(cfg['training']['epochs']):
        model.train()
        for batch in train_loader:
            x = batch['x'].to(device)
            y = batch['y'].to(device)
            pred = model(x)
            loss, _ = total_loss(model, x, pred, y, cfg)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: Loss = {loss.item():.6f}")
    
    print("Training complete!")

if __name__ == "__main__":
    main()
'''

with open('train.py', 'w') as f:
    f.write(train_script)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = f'/content/drive/MyDrive/srpsi-colab-baseline/runs/v1_baseline_{timestamp}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ 训练脚本已创建")
print(f"📁 输出目录: {OUTPUT_DIR}")
print(f"🚀 准备开始训练...")

In [ ]:
# Step 6: 开始训练
!python train.py